In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.io as spio
import warnings
import sys
import time
import import_ipynb
import main_method

def createBehDict(aBehData,ofcBoolean):
    correctInt = []
    taskType = []
    startArm = []
    goalArm = []
    for i in range(np.shape(aBehData)[0]):
        if ofcBoolean == True:
            currTrialBeh = aBehData[i]
            correctInt.append(int(float((currTrialBeh[0])[:3])))
            taskType.append('S')
            startArm.append((currTrialBeh[1])[0])
            goalArm.append((currTrialBeh[2])[0])
        else:
            currTrialBeh = aBehData[i]
            correctInt.append(int(currTrialBeh[0]))
            taskType.append((currTrialBeh[1])[0])
            startArm.append((currTrialBeh[3])[0])
            goalArm.append((currTrialBeh[4])[0])
    behDictSplit = {'trialCorrect':correctInt,
                    'taskType':taskType,
                    'startArm':startArm,
                    'goalArm':goalArm,
                   }
    return behDictSplit

def padOutVals(cellArr):
    C = np.squeeze(cellArr)
    maxPadVal = 0
    for i in range(len(C)):
        if np.shape(C[i])[1] > maxPadVal:
            maxPadVal = np.shape(C[i])[1]
    for g in range(len(C)):
        padNumber = abs(maxPadVal - np.shape(C[g])[1])
        C[g] = np.pad(C[g], ((0,0),(0,padNumber)), 'constant', constant_values=0)
    Cupd = np.nan_to_num(C,nan=0.,posinf=0.,neginf=0.)
    return Cupd

def loadFromSmith(gFileOfInterest):
    gmat = spio.loadmat(gFileOfInterest)
    return gmat.get('lowGra').flatten(),gmat.get('midGra').flatten(),gmat.get('uppGra').flatten()

def cleanUpStartInd(dictToClean):
    cleanedDict = dictToClean
    newStartInds = []
    for i in range(np.shape(dictToClean['startPosInd'])[0]):
        val = (((dictToClean['startPosInd'])[i])[0])[0]
        newStartInds.append(int(val))
    cleanedDict['startPosInd'] = newStartInds
    return cleanedDict

def removeCueTrials(xDict):
    newDict = {}
    taskType = xDict['taskType']
    taskTypeInds = []
    for z in range(np.shape(taskType)[0]):
        if taskType[z] == 'S':
            taskTypeInds.append(z)
    for xKey in xDict.keys():
        if xKey != 'taskType':
            if xKey == 'regionIndex' or 'Gra' in xKey or xKey == 'scale':
                newDict.update({xKey:xDict[xKey]})
            else:
                newDict.update({xKey:np.take(xDict[xKey],taskTypeInds,axis=0)})
    return newDict

def removeSpaTrials(xDict):
    newDict = {}
    taskType = xDict['taskType']
    taskTypeInds = []
    for z in range(np.shape(taskType)[0]):
        if taskType[z] == 'C':
            taskTypeInds.append(z)
    for xKey in xDict.keys():
        if xKey != 'taskType':
            if xKey == 'regionIndex' or 'Gra' in xKey or xKey == 'scale':
                newDict.update({xKey:xDict[xKey]})
            else:
                newDict.update({xKey:np.take(xDict[xKey],taskTypeInds,axis=0)})
    return newDict

def getStartingTrialTimes(zDict):
    startingDict = {}
    for zkey in zDict.keys():
        if zkey != 'trialTime' and zkey != 'startPosInd':
            startingDict.update({zkey:zDict[zkey]})
    trialStartTimes = []
    trialTimeLen = []
    trialTimeArr = [None]
    startPosInd = zDict['startPosInd']
    trialTimes = zDict['trialTime']
    for tr in range(np.shape(startPosInd)[0]):
        currStartPosInd = startPosInd[tr]
        currTrialTimes = trialTimes[tr]
        currTrialTimeArr = currTrialTimes[int(currStartPosInd):]
        timeOfInterest = currTrialTimes[int(currStartPosInd)]
        trialTimeOfInterest = (currTrialTimes[-1] - currTrialTimes[int(currStartPosInd)])/1000000
        currTrialTimeArr = (currTrialTimes[int(currStartPosInd):] - currTrialTimes[int(currStartPosInd)])/1000000
        trialStartTimes.append(timeOfInterest)
        trialTimeLen.append(trialTimeOfInterest)
        trialTimeArr.append(currTrialTimeArr)
    startingDict.update({'startingTrialTimes':np.asarray(trialStartTimes),
                         'timeOfTrial':np.asarray(trialTimeLen),
                         'trialTimeArr':trialTimeArr[1:],
                        })
    return startingDict

def normalizeSpikeTimes(yDict):
    normedSpikeTimeDict = {}
    for ykey in yDict.keys():
        if ykey != 'spikeTimes' and ykey != 'startingTrialTimes':
            normedSpikeTimeDict.update({ykey:yDict[ykey]})
    normSpikeTimes = []
    spikeTimes = yDict['spikeTimes']
    startingTrialTimes = yDict['startingTrialTimes']
    trialDuration = yDict['timeOfTrial']
    for tr in range(np.shape(spikeTimes)[0]):
        currSpikeTimes = np.nan_to_num(spikeTimes[tr],nan=0,posinf=0,neginf=0)
        currNormSpTime = ((currSpikeTimes - startingTrialTimes[tr])/1000000)
        resNormSpikes = np.reshape(currNormSpTime,(1,np.shape(currNormSpTime)[0],np.shape(currNormSpTime)[1]))
        if tr == 0:
            normSpikeTimes = resNormSpikes
        else:
            normSpikeTimes = np.concatenate((normSpikeTimes,resNormSpikes),axis=0)
    normedSpikeTimeDict.update({'spikeTimes':normSpikeTimes})
    return normedSpikeTimeDict

def getSplitInds(regInds):
    hpcInds = []
    pfcInds = []
    for z in range(np.shape(regInds)[0]):
        if regInds[z] == 1:
            hpcInds.append(z)
        else: 
            pfcInds.append(z)
    return hpcInds,pfcInds

def createBinnedSpikeTimeArr(aDict):
    spikeCellArrDict = {}
    for akey in aDict.keys():
        if akey != 'spikeTimes' and akey != 'regionIndex':
            spikeCellArrDict.update({akey:aDict[akey]})
    hInds,pInds = getSplitInds(aDict['regionIndex'])
    spikeTimes = aDict['spikeTimes']
    trialTimes = aDict['timeOfTrial']
    arrByUnitHPC = [None]
    arrByUnitPFC = [None]
    arrByUnitAll = [None]
    arrTimeBins = [None]
    for tr in range(np.shape(spikeTimes)[0]):
        currSpikeTimes = spikeTimes[tr]
        currTrialTimes = round(trialTimes[tr],2)
        currBins = np.arange(0,currTrialTimes+.125,.125) #Samples every 125 ms, alter as needed
        trialSpikeHist = []
        for un in range(np.shape(currSpikeTimes)[0]):
            currUnitTimes = currSpikeTimes[un]
            histArr,binEdges = np.histogram(currUnitTimes,bins=currBins,range=(0,currTrialTimes))
            histArrRes = np.reshape(histArr,(np.shape(histArr)[0],1))
            if un == 0:
                trialSpikeHist = histArrRes
            else:
                trialSpikeHist = np.concatenate((trialSpikeHist,histArrRes),axis=1)
        trialSpikeHistRes = np.reshape(trialSpikeHist,(np.shape(trialSpikeHist)[0],1,np.shape(trialSpikeHist)[1]))
        trialSpikeHistUpd = np.where(trialSpikeHistRes > 1,1,trialSpikeHistRes).astype(int)
        hpcTrialSpikeHist = np.take(trialSpikeHistUpd,hInds,axis=2)
        pfcTrialSpikeHist = np.take(trialSpikeHistUpd,pInds,axis=2)
        arrByUnitHPC.append(np.nan_to_num(hpcTrialSpikeHist,nan=0,posinf=0,neginf=0))
        arrByUnitPFC.append(np.nan_to_num(pfcTrialSpikeHist,nan=0,posinf=0,neginf=0))
        arrByUnitAll.append(np.nan_to_num(trialSpikeHistUpd,nan=0,posinf=0,neginf=0))
        arrTimeBins.append(currBins)
    spikeCellArrDict.update({'spikeHistHPC':arrByUnitHPC[1:],'spikeHistPFC':arrByUnitPFC[1:],
                             'spikeHistAll':arrByUnitAll[1:],'timeBins':arrTimeBins[1:],
                            })
    return spikeCellArrDict            

def getStartChoiceInds(zDict):
    updIndDict = {}
    for zkey in zDict.keys():
        if zkey != 'distTrav' and zkey != 'timeBins' and zkey != 'timeOfTrial' and zkey != 'trialTimeArr':
            updIndDict.update({zkey:zDict[zkey]})
    distTrav = zDict['distTrav']
    trialTimeArr = zDict['trialTimeArr']
    timeBins = zDict['timeBins']
    startInds = []
    choiceInds = []
    goalInds = []
    for tr in range(np.shape(distTrav)[0]):
        currDistTrav = distTrav[tr]
        currTrialTimeArr = trialTimeArr[tr]
        currTimeBins = timeBins[tr]
        startDistInd = np.argmax(currDistTrav > 20)
        choiceDistInd = np.argmax(currDistTrav > 58)
        goalDistInd = np.argmax(currDistTrav > 71)
        currStartTime = currTrialTimeArr[int(startDistInd)]
        currChoiceTime = currTrialTimeArr[int(choiceDistInd)]
        currGoalTime = currTrialTimeArr[int(goalDistInd)]
        startInds.append(np.argmax(currTimeBins > currStartTime))
        choiceInds.append(np.argmax(currTimeBins > currChoiceTime))
        goalInds.append(np.argmax(currTimeBins > currGoalTime))
    updIndDict.update({'startInds':np.asarray(startInds).astype(int),
                       'choiceInds':np.asarray(choiceInds).astype(int),
                       'goalInds':np.asarray(goalInds).astype(int),
                      })
    return updIndDict

def runSSASC(checkSpikes,toScaleTo):
    #Error handling, if any failures in the emd method - 
    #immediately returns empty cov matrix array and boolean to later remove the trial
    try:
        emd = main_method.run(checkSpikes.astype(int),int(2),map_function='cg',lmbda1=int(100),lmbda2=int(200),
                          param_est='pseudo', param_est_eta='bethe_hybrid',
                         )
        emdValsOut = emd.sigma_o
        emdMatOut = []
        for ts in range(np.shape(emdValsOut)[0]):
            currEmdVals = (emdValsOut[ts])[int(emd.N):]
            idx = np.tril_indices(emd.N, k=-1, m=emd.N)
            ltMat = np.zeros((int(emd.N),int(emd.N)))
            ltMat[idx] = currEmdVals
            updMatrix = ltMat + np.flip(np.flip(ltMat,axis=0),axis=1)
            rowDiag,colDiag = np.diag_indices(np.shape(updMatrix)[0],ndim=2)
            updMatrix[rowDiag,colDiag] = (emdValsOut[ts])[:int(emd.N)]
            updMatrixRes = np.reshape(updMatrix,(1,int(emd.N),int(emd.N)))
            if ts == 0:
                emdMatOut = updMatrixRes
            else:
                emdMatOut = np.concatenate((emdMatOut,updMatrixRes),axis=0)
        return (emdMatOut*toScaleTo).astype(int), 1
    except:
        return [], 0

def buildCovarianceMatrix(inDict,scaleDigit):
    newUpdDict = {}
    for ikey in inDict.keys():
        if ikey != 'spikeHistHPC' and ikey != 'spikeHistPFC':
            newUpdDict.update({ikey:inDict[ikey]})
    spikeHistHPC = inDict['spikeHistHPC']
    spikeHistPFC = inDict['spikeHistPFC']
    spikeHistAll = inDict['spikeHistAll']
    covMatHPC = [None]
    keepMatHPC = [None]
    covMatPFC = [None]
    keepMatPFC = [None]
    covMatAll = [None]
    keepMatAll = [None]
    for tr in range(np.shape(spikeHistHPC)[0]):
        currAll = spikeHistAll[tr]
        allEmdOut,keepBoolAll = runSSASC(currAll,scaleDigit)
        covMatAll.append(allEmdOut)
        keepMatAll.append(keepBoolAll)
    newUpdDict.update({'allCovMat':covMatAll[1:],'boolMatAll':keepMatAll[1:],
                      })
    return newUpdDict

def loadAndProcessValues(fileOfInterest,graFileOfInterest):
    if '/OFC_LFR/' in fileOfInterest:
        boolOFC = True
    else:
        boolOFC = False
    #lowBound,midBound,uppBound = loadFromSmith(graFileOfInterest)
    mat = spio.loadmat(fileOfInterest)
    allBehData = mat.get('behData')
    behDict = createBehDict(allBehData,boolOFC)
    behDict.update({'scale':1000, #Alter as needed, e.g. value of 10000 rounds correlations to 4 decimal places
                    'regionIndex':mat.get('regionIndex').flatten(),
                    'startPosInd':mat.get('startPosInd').flatten(),
                    'trialTime':mat.get('trialTime'),
                    'spikeTimes':mat.get('spikeTimes'),
                    'distTrav':mat.get('distTrav'),
                   })
    for key in behDict.keys():
        if key != 'taskType' and key != 'trialCorrect' and key != 'startArm' and key != 'goalArm':
            if key != 'regionIndex' and key != 'startPosInd' and key != 'scale':
                if key == 'spikeTimes':
                    if len(np.shape(behDict[key])) == 3:
                        behDict.update({f'{key}':behDict[key]})
                    else:
                        C = padOutVals(np.squeeze(behDict[key]))
                        X = []
                        for i in range(len(C)):
                            X.append(C[i])
                        behDict.update({f'{key}':X})
                else:
                    C = np.squeeze(behDict[key])
                    Y = []
                    for i in range(len(C)):
                        C[i] = C[i].flatten()
                        Y.append(C[i])
                    behDict.update({f'{key}':Y})
    #behDict.update({'lowGra':lowBound,'midGra':midBound,'uppGra':uppBound})
    dictOfInterestClean = cleanUpStartInd(behDict)
    spatialOnlyDict = removeCueTrials(dictOfInterestClean)
    startingTrialTimesDict = getStartingTrialTimes(spatialOnlyDict)
    spikeTimeNormedDict = normalizeSpikeTimes(startingTrialTimesDict)
    binnedSpikeTimeDict = createBinnedSpikeTimeArr(spikeTimeNormedDict)
    startChoiceIndDict = getStartChoiceInds(binnedSpikeTimeDict)
    covMatDict = buildCovarianceMatrix(startChoiceIndDict,startChoiceIndDict['scale'])
    return covMatDict

def pathWalk(rawPs,prbPs,savPs,checkList):
    for z in range(len(rawPs)):
        rawP = rawPs[z]
        prbP = prbPs[z]
        savP = savPs[z]
        for zRoot,zDirs,zFiles in os.walk(rawP):
            for zFile in zFiles:
                if zFile.endswith('.mat') and zFile not in checkList:
                    currRawFile = os.path.join(rawP,zFile)
                    currPrbFile = os.path.join(prbP,zFile)
                    currSavFile = os.path.join(savP,zFile)
                    dictToSave = loadAndProcessValues(currRawFile,currPrbFile)
                    saveVar = spio.savemat(currSavFile,dictToSave)
        #print(f'Built from {rawP}')
    return 'Done'

pathRaw = []

pathPrb = []

pathSav = []

chList = []

pathWalk(pathRaw,pathPrb,pathSav,chList)
